In [1]:
import sys
sys.path.append("..")
from src.features import *
from src.data.make_dataset import *
from src.data.weights import *
from src.features.zoning_nonconformity_scripts import *

c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(


In [2]:
par_gdf = get_landuse_data("Newton")
zon_gdf = get_zoning_data("Newton")


In [3]:
merge_test= zoning_merge(zoning_gdf= zon_gdf, parcels_gdf= par_gdf)

Split Zoned Parcels Detected
Clean Join successful?
True
Zone Code succesfully joined?
False
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 23947 entries, 0 to 23946
Data columns (total 64 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   LOC_ID              23947 non-null  object  
 1   geometry            23947 non-null  geometry
 2   Unnamed: 0          23947 non-null  int64   
 3   TOWN_ID             23947 non-null  int64   
 4   PROP_ID             23947 non-null  object  
 5   BLDG_VAL            23947 non-null  int64   
 6   LAND_VAL            23947 non-null  int64   
 7   OTHER_VAL           23947 non-null  int64   
 8   TOTAL_VAL           23947 non-null  int64   
 9   FY                  23947 non-null  object  
 10  LOT_SIZE            23784 non-null  float64 
 11  LS_DATE             23947 non-null  object  
 12  LS_PRICE_S          23947 non-null  float64 
 13  LS_PRICE_L          23947 non-null 

In [4]:
parcels_gdf = par_gdf
print("Rows: ", len(parcels_gdf['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(parcels_gdf['LOC_ID'])))
zoning_gdf = zon_gdf

par_zon_join = parcels_gdf.overlay(zoning_gdf, how = "intersection", keep_geom_type = True)
print("Overlay")
print(len(par_zon_join['LOC_ID']) >  len(set(par_zon_join['LOC_ID'])))
print("Rows: ", len(par_zon_join['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(par_zon_join['LOC_ID'])))

par_zon_sjoin= parcels_gdf.sjoin(zoning_gdf)
print("sjoin")
print(len(par_zon_sjoin['LOC_ID']) >  len(set(par_zon_sjoin['LOC_ID'])))
print("Rows: ", len(par_zon_sjoin['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(par_zon_sjoin['LOC_ID'])))



Rows:  23947
Unique LOCIDS:  23947
Overlay
True
Rows:  24068
Unique LOCIDS:  23772
sjoin
True
Rows:  29787
Unique LOCIDS:  23863


In [5]:

print ("Split Zoned Parcels Detected")       
# determine the proportion of the total parcel area
par_zon_join['zone_area'] = par_zon_join['geometry'].area
par_zon_join['zone_share'] = par_zon_join.apply(lambda row: row['zone_area']*10.7639/row['LOT_SIZE_GIS'], axis=1) #its a row

# ID the index of the row with the largest share of a parcel in a zone for each unique LOCID, reset_index() makes into a df
idx = par_zon_join.fillna(999999).groupby('LOC_ID')['zone_share'].idxmax().reset_index()

# subset the original spatial join to the rows where the share is the largest for each unique LOC ID--should be one row for each LOCID again
clean_join = par_zon_join.loc[idx['zone_share']]
print("Clean Join successful?")
print(len(idx) == len(clean_join))
print("Index: ", len(idx))
print("Clean Join", len(clean_join))
# cut that table to just the LOC ID and the ZO Code, confirm pd
par_zon_xwalk = clean_join[['LOC_ID','ZO_CODE']]
# join the zone code onto the parcels, double check the join
parcels_zone_rec = pd.merge(parcels_gdf, par_zon_xwalk, left_on= 'LOC_ID', right_on= "LOC_ID", how = "left")
print("Zone Code succesfully joined?")
print(len(parcels_zone_rec['LOC_ID']) == len(clean_join))

print("Zoning Reconciled Parcel:", len(parcels_zone_rec))

# take the zoning input and get rid of the geometry so we can do non-spatial joins
zoning_table = pd.DataFrame(zoning_gdf.drop(columns= 'geometry'))
# join the rest of the zoning table back to the parcels
par_zon_join_fixed = pd.merge(parcels_zone_rec, zoning_table, left_on= 'ZO_CODE', right_on= 'ZO_CODE', how = "inner")
print("Zones assigned to Parcels by Largest Share")


Split Zoned Parcels Detected


Clean Join successful?
True
Index:  23772
Clean Join 23772
Zone Code succesfully joined?
False
Zoning Reconciled Parcel: 23947
Zones assigned to Parcels by Largest Share


In [10]:
import gc

gc.collect()

boston_parcels = "bye"

In [ ]:
par_zon_join_fixed

In [ ]:
# merge_test['zo_code'].unique()
len(zon_gdf)
print("Rows: ", len(merge_test['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(merge_test['LOC_ID'])))


Unique LOCIDS:  23772
